# 01 · Perturb-seq data acquisition & dataset audit
**Goal:** obtain the Replogle K562 (essential-gene CRISPRi) dataset, store it on Google Drive, and audit size, structure, controls and feasibility.

**Experiment in one paragraph.** Each cell received a guide RNA (sgRNA) directing a catalytically-dead Cas9 fused to a repressor (CRISPRi) to a target gene's promoter, which silences that gene. Single-cell RNA-seq then measures the whole transcriptome of each cell. *Non-targeting* guides give control cells. So every cell has (i) an expression vector and (ii) a perturbation label.

Set `SYNTHETIC = True` in the header cell for a quick smoke test without downloading anything.

In [ ]:
# ---- Bootstrap: Colab (Google Drive) or local ----
import os, sys, subprocess
REPO_URL = "https://github.com/<your-username>/PerturbAI.git"   # <-- edit once after you create the repo
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/PerturbAI"):
        subprocess.run(["git", "clone", REPO_URL, "/content/PerturbAI"], check=True)
    os.chdir("/content/PerturbAI")
    subprocess.run(["git", "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
else:
    try:
        import perturbai  # installed with `pip install -e .`
    except ImportError:
        sys.path.insert(0, os.path.abspath("../src"))  # fallback when running from notebooks/
from perturbai.paths import get_paths
P = get_paths()
print("Data root:", P.root)

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp, matplotlib.pyplot as plt
from perturbai.paths import artifact
from perturbai.utils import set_seed, load_config
CFG = load_config("data_config.yaml"); MCFG = load_config("model_config.yaml")
set_seed(CFG["seed"])
SYNTHETIC = False        # True = 1-minute smoke test on simulated data (no download)
DS = "replogle_k562_essential"
INFO = CFG["datasets"][DS]
PERT, CTRL, BATCH = INFO["perturbation_col"], INFO["control_label"], INFO["batch_col"]
TAG = DS + ("_synthetic" if SYNTHETIC else "")   # artifact prefix

In [ ]:
from perturbai.utils import memory_report
memory_report()

## Load (local Drive file → Zenodo download → pertpy fallback)
The first run downloads the file into `<data root>/raw/<dataset>/`; later sessions reuse it.

In [ ]:
from perturbai.data.loaders import load_dataset, detect_columns, audit_adata, save_audit
adata = load_dataset(DS, P, synthetic=SYNTHETIC)
adata

## Identify perturbation labels and control cells

In [ ]:
found = detect_columns(adata)
print(found["pert_col"], "|", found["control_label"])
assert found["pert_col"] == PERT and found["control_label"] == CTRL, (
    "Edit configs/data_config.yaml so perturbation_col / control_label match your file: " + str(found))
adata.obs.head()

## Audit

In [ ]:
import json
audit = audit_adata(adata, PERT, CTRL)
print(json.dumps(audit, indent=2, default=str))
print("Saved:", save_audit(audit, DS, P))

In [ ]:
vc = adata.obs[PERT].astype(str).value_counts()
per = vc.drop(CTRL, errors="ignore")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(per.values, bins=60); ax[0].set(xlabel="cells per perturbation", ylabel="# perturbations", title="Coverage")
ax[0].axvline(50, ls="--", c="r")
ax[1].bar(["control", "perturbed"], [vc.get(CTRL, 0), per.sum()]); ax[1].set_title("Cells: control vs perturbed")
plt.tight_layout(); plt.show()
print(per.describe())

## Is X raw counts or already normalised? (decides notebook 03 behaviour)

In [ ]:
from perturbai.data.loaders import looks_like_counts
print("looks like raw integer counts:", looks_like_counts(adata.X))
print("layers:", list(adata.layers))
print("target genes present in var:", pd.Index(per.index).isin(adata.var_names).sum(), "of", len(per))

## Feasibility report (write your conclusions here)
- Memory of `X` (GB) vs Colab RAM (`memory_report()`):
- Plan: work on a subset of perturbations (config `subset:`), then scale up.
- Red flags (perturbations with <50 cells, targets absent from `var`):